# Branch 3 — Combined Numerical + Textual Models

Fusion experiments using the **same numerical features and SEC filing text on identical rows**. Each combined representation is fed into L2 logistic regression. Includes Numerical+TF-IDF, Numerical+FinBERT, Numerical+Sentence Transformer, and Numerical+Sentence Transformer+FinBERT.

In [ ]:
!pip -q install pyarrow openpyxl xgboost sentence-transformers transformers accelerate beautifulsoup4 requests tqdm

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, log_loss, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH: Path | None = None
NEUTRAL_BAND = 0.02
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
THRESHOLD_GRID = np.linspace(0.20, 0.80, 121)


OUTPUT_DIR = Path('/content/semiconductor_branch_combined')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DATA_PATH: Path | None = None
SEC_USER_AGENT = 'YOUR NAME your.email@example.com'
DOWNLOAD_SEC_TEXT_FROM_EDGAR = True
TEXT_EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
MIN_TEXT_CHARS = 500
MAX_DOCUMENT_CHARS = 120_000
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
MAX_CHUNKS_PER_FILING = 16
TEXT_BATCH_SIZE = 32
TEXT_PCA_COMPONENTS = 32
RUN_FINBERT_SENTIMENT = True
FINBERT_MODEL_NAME = 'ProsusAI/finbert'
TEXT_CACHE_DIR = OUTPUT_DIR/'sec_text_cache'
TEXT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

## Load the same dataset used by v6.1

In [ ]:
def discover_data_path() -> Path:
    candidates = [
        Path('/content/drive/MyDrive/sec_research_semiconductor/semiconductor_sec_numeric_text.parquet'),
        Path('/content/drive/MyDrive/sec_research_semiconductor/sec_experiment_semiconductor.parquet'),
        Path('/content/semiconductor_sec_numeric_text.parquet'),
        Path('/content/sec_experiment_semiconductor.parquet'),
        Path('semiconductor_sec_numeric_text.parquet'),
        Path('semiconductor_sec_numeric_text.csv'),
        Path('sec_experiment_semiconductor.parquet'),
        Path('model_dataset.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            print('Found dataset automatically:', candidate)
            return candidate

    try:
        from google.colab import files
        print('Upload the same parquet/CSV dataset used by v6.1.')
        uploaded = files.upload()
        choices = [Path(name) for name in uploaded if Path(name).suffix.lower() in {'.parquet', '.csv'}]
        if not choices:
            raise FileNotFoundError('Upload a .parquet or .csv file.')
        return choices[0]
    except ImportError as exc:
        raise FileNotFoundError('Set DATA_PATH to the v6.1 dataset path.') from exc

resolved_path = DATA_PATH if DATA_PATH is not None else discover_data_path()
if resolved_path.suffix.lower() == '.parquet':
    raw = pd.read_parquet(resolved_path)
elif resolved_path.suffix.lower() == '.csv':
    raw = pd.read_csv(resolved_path)
else:
    raise ValueError('Use a .parquet or .csv dataset.')

print('Loaded:', resolved_path)
print('Rows:', len(raw), '| Columns:', len(raw.columns))

## Rebuild target exactly as v6.1

In [ ]:
data = raw.copy()

# Standardize identifiers and dates.
if "cik" not in data.columns:
    if "ticker" not in data.columns:
        raise ValueError("The dataset must contain either cik or ticker.")
    data["cik"] = data["ticker"].astype(str)

if "ticker" not in data.columns:
    data["ticker"] = data["cik"].astype(str)

date_candidates = [
    "quarter_end",
    "feature_cutoff_date",
    "label_available_date",
]
for column in date_candidates:
    if column in data.columns:
        data[column] = pd.to_datetime(data[column], errors="coerce")

if "quarter_end" not in data.columns:
    raise ValueError("The dataset must contain quarter_end.")

data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

# Map names from the earlier proof-of-concept dataset when needed.
rename_aliases = {
    "revenue_mm": "revenue",
    "inventory_mm": "inventory",
    "accounts_receivable_mm": "accounts_receivable",
    "operating_cash_flow_mm": "operating_cash_flow",
    "inventory_ratio": "inventory_to_ttm_revenue",
    "ar_ratio": "receivables_to_ttm_revenue",
    "ocf_margin": "cash_flow_margin",
}
for old_name, new_name in rename_aliases.items():
    if new_name not in data.columns and old_name in data.columns:
        data[new_name] = pd.to_numeric(data[old_name], errors="coerce")

numeric_candidates = [
    "revenue",
    "revenue_yoy_growth",
    "revenue_momentum",
    "next_quarter_growth",
    "gross_margin",
    "operating_margin",
    "cash_flow_margin",
    "inventory",
    "inventory_to_ttm_revenue",
    "accounts_receivable",
    "receivables_to_ttm_revenue",
    "capital_expenditures",
    "capex_to_revenue",
    "total_assets",
    "log_assets",
    "liabilities_to_assets",
]
for column in numeric_candidates:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Reconstruct YoY growth if it is missing and revenue is available.
if "revenue_yoy_growth" not in data.columns and "revenue" in data.columns:
    revenue_lag4 = grouped["revenue"].shift(4)
    quarter_lag4 = grouped["quarter_end"].shift(4)
    gap4 = (data["quarter_end"] - quarter_lag4).dt.days
    data["revenue_yoy_growth"] = np.where(
        gap4.between(320, 410),
        data["revenue"] / revenue_lag4 - 1.0,
        np.nan,
    )

# Reconstruct current momentum if missing.
if "revenue_momentum" not in data.columns:
    previous_growth = grouped["revenue_yoy_growth"].shift(1)
    previous_end = grouped["quarter_end"].shift(1)
    gap1 = (data["quarter_end"] - previous_end).dt.days
    data["revenue_momentum"] = np.where(
        gap1.between(60, 125),
        data["revenue_yoy_growth"] - previous_growth,
        np.nan,
    )

# Reconstruct next-quarter growth if missing.
if "next_quarter_growth" not in data.columns:
    next_growth = grouped["revenue_yoy_growth"].shift(-1)
    next_end = grouped["quarter_end"].shift(-1)
    next_gap = (next_end - data["quarter_end"]).dt.days
    data["next_quarter_growth"] = np.where(
        next_gap.between(60, 125),
        next_growth,
        np.nan,
    )

data["future_growth_change"] = (
    data["next_quarter_growth"] - data["revenue_yoy_growth"]
)

data["target_clean"] = pd.Series(
    np.select(
        [
            data["future_growth_change"] > NEUTRAL_BAND,
            data["future_growth_change"] < -NEUTRAL_BAND,
        ],
        [1.0, 0.0],
        default=np.nan,
    ),
    index=data.index,
)

data["target_status"] = np.select(
    [
        data["future_growth_change"] > NEUTRAL_BAND,
        data["future_growth_change"] < -NEUTRAL_BAND,
        data["future_growth_change"].abs() <= NEUTRAL_BAND,
    ],
    ["Accelerating", "Decelerating", "Neutral"],
    default="Unavailable",
)

target_summary = (
    data["target_status"]
    .value_counts(dropna=False)
    .rename_axis("target_status")
    .reset_index(name="rows")
)
target_summary["fraction"] = target_summary["rows"] / len(data)
display(target_summary)

## Financial feature engineering

In [ ]:
data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

previous_end = grouped["quarter_end"].shift(1)
gap1 = (data["quarter_end"] - previous_end).dt.days
valid_qoq = gap1.between(60, 125)

quarter_lag4 = grouped["quarter_end"].shift(4)
gap4 = (data["quarter_end"] - quarter_lag4).dt.days
valid_yoy = gap4.between(320, 410)

def add_qoq_change(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(1)
        data[output] = np.where(valid_qoq, data[column] - lagged, np.nan)

def add_yoy_growth(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(4)
        data[output] = np.where(
            valid_yoy & (lagged.abs() > 1e-12),
            data[column] / lagged - 1.0,
            np.nan,
        )

if "revenue" in data.columns:
    revenue_lag1 = grouped["revenue"].shift(1)
    data["sequential_revenue_growth"] = np.where(
        valid_qoq & (revenue_lag1.abs() > 1e-12),
        data["revenue"] / revenue_lag1 - 1.0,
        np.nan,
    )

add_qoq_change("gross_margin", "gross_margin_change")
add_qoq_change("operating_margin", "operating_margin_change")
add_qoq_change("cash_flow_margin", "cash_flow_margin_change")
add_qoq_change(
    "inventory_to_ttm_revenue",
    "inventory_to_ttm_revenue_change",
)
add_qoq_change(
    "receivables_to_ttm_revenue",
    "receivables_to_ttm_revenue_change",
)
add_qoq_change("capex_to_revenue", "capex_to_revenue_change")

add_yoy_growth("inventory", "inventory_yoy_growth")
add_yoy_growth("accounts_receivable", "receivables_yoy_growth")
add_yoy_growth("capital_expenditures", "capex_yoy_growth")

if "inventory_yoy_growth" in data.columns:
    data["inventory_revenue_growth_gap"] = (
        data["inventory_yoy_growth"] - data["revenue_yoy_growth"]
    )

if "receivables_yoy_growth" in data.columns:
    data["receivables_revenue_growth_gap"] = (
        data["receivables_yoy_growth"] - data["revenue_yoy_growth"]
    )

data["calendar_quarter"] = data["quarter_end"].dt.to_period("Q")

relative_base_features = [
    column
    for column in [
        "revenue_yoy_growth",
        "revenue_momentum",
        "sequential_revenue_growth",
        "gross_margin",
        "gross_margin_change",
        "cash_flow_margin",
        "inventory_to_ttm_revenue",
        "inventory_revenue_growth_gap",
        "receivables_to_ttm_revenue",
        "capex_to_revenue",
    ]
    if column in data.columns
]

quarter_medians = (
    data.groupby("calendar_quarter")[relative_base_features]
    .median()
    .sort_index()
)
prior_quarter_medians = quarter_medians.shift(1).add_suffix(
    "_prior_sector_median"
)

data = data.merge(
    prior_quarter_medians,
    left_on="calendar_quarter",
    right_index=True,
    how="left",
)

for feature in relative_base_features:
    median_column = f"{feature}_prior_sector_median"
    data[f"{feature}_relative_to_sector"] = (
        data[feature] - data[median_column]
    )

engineered_features = [
    column
    for column in data.columns
    if (
        column.endswith("_change")
        or column.endswith("_yoy_growth")
        or column.endswith("_growth_gap")
        or column.endswith("_relative_to_sector")
        or column == "sequential_revenue_growth"
    )
]

print("Engineered features:", len(engineered_features))
display(data[["ticker", "quarter_end"] + engineered_features[:12]].head(10))

## Chronological split

In [ ]:
# Use an existing split only when it contains all three required groups.
required_splits = {"train", "validation", "test"}

existing_splits = (
    set(data["split"].dropna().astype(str).str.lower().unique())
    if "split" in data.columns
    else set()
)

existing_split_is_usable = required_splits.issubset(existing_splits)

if existing_split_is_usable:
    data["split"] = data["split"].astype(str).str.lower()
    print("Using the existing train/validation/test split.")
else:
    if "split" in data.columns:
        print(
            "The existing split column does not contain all three groups. "
            "Rebuilding the split chronologically."
        )

    data["split"] = pd.NA

    # Use the most conservative available timestamp:
    # when the target became knowable, then the feature cutoff, then quarter end.
    if (
        "label_available_date" in data.columns
        and data["label_available_date"].notna().any()
    ):
        split_date_column = "label_available_date"
    elif (
        "feature_cutoff_date" in data.columns
        and data["feature_cutoff_date"].notna().any()
    ):
        split_date_column = "feature_cutoff_date"
    else:
        split_date_column = "quarter_end"

    eligible_dates = (
        data.loc[data["target_clean"].notna(), split_date_column]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(eligible_dates) < 3:
        raise ValueError(
            "At least three distinct dated periods are required to create "
            "train, validation, and test splits."
        )

    # Automatic chronological 60% / 20% / 20% split by distinct dates.
    train_position = max(0, min(len(eligible_dates) - 3, int(len(eligible_dates) * 0.60) - 1))
    validation_position = max(
        train_position + 1,
        min(len(eligible_dates) - 2, int(len(eligible_dates) * 0.80) - 1),
    )

    automatic_train_end = eligible_dates.iloc[train_position]
    automatic_validation_end = eligible_dates.iloc[validation_position]

    labeled = data["target_clean"].notna()
    split_dates = data[split_date_column]

    data.loc[
        labeled & (split_dates <= automatic_train_end),
        "split",
    ] = "train"

    data.loc[
        labeled
        & (split_dates > automatic_train_end)
        & (split_dates <= automatic_validation_end),
        "split",
    ] = "validation"

    data.loc[
        labeled & (split_dates > automatic_validation_end),
        "split",
    ] = "test"

    print("Split date column:", split_date_column)
    print("Automatic train end:", automatic_train_end)
    print("Automatic validation end:", automatic_validation_end)

model_data = data[
    data["target_clean"].notna()
    & data["split"].isin(["train", "validation", "test"])
].copy()
model_data["target_clean"] = model_data["target_clean"].astype(int)

split_summary = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        companies=("cik", "nunique"),
        first_quarter=("quarter_end", "min"),
        last_quarter=("quarter_end", "max"),
        acceleration_rate=("target_clean", "mean"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
display(split_summary)

available_splits = set(model_data["split"].dropna().unique())
missing_splits = required_splits - available_splits
if missing_splits:
    raise ValueError(
        f"Could not create these splits: {sorted(missing_splits)}. "
        "The dataset may have too few labeled dates after applying the "
        "neutral band."
    )

for split_name in ["train", "validation", "test"]:
    split_frame = model_data[model_data["split"] == split_name]
    if split_frame["target_clean"].nunique() < 2:
        print(
            f"Warning: {split_name} contains only one target class after "
            f"applying the {NEUTRAL_BAND:.1%} neutral band. "
            "Try reducing NEUTRAL_BAND to 0.01 if model fitting fails."
        )

## Numerical feature list

In [ ]:
candidate_numeric_features = [
    'revenue_yoy_growth', 'revenue_momentum', 'sequential_revenue_growth',
    'gross_margin', 'gross_margin_change', 'operating_margin',
    'operating_margin_change', 'cash_flow_margin', 'cash_flow_margin_change',
    'inventory_to_ttm_revenue', 'inventory_to_ttm_revenue_change',
    'inventory_yoy_growth', 'inventory_revenue_growth_gap',
    'receivables_to_ttm_revenue', 'receivables_to_ttm_revenue_change',
    'receivables_yoy_growth', 'receivables_revenue_growth_gap',
    'capex_to_revenue', 'capex_to_revenue_change', 'capex_yoy_growth',
    'log_assets', 'liabilities_to_assets',
]

candidate_numeric_features += [
    f'{feature}_relative_to_sector' for feature in relative_base_features
]

numeric_features = [
    c for c in dict.fromkeys(candidate_numeric_features)
    if c in model_data.columns
    and model_data[c].notna().sum() >= 10
    and model_data[c].nunique(dropna=True) > 1
]

categorical_features = [
    c for c in ['fiscal_quarter', 'semiconductor_subgroup']
    if c in model_data.columns and model_data[c].notna().any()
]
feature_columns = numeric_features + categorical_features
print('Numerical features:', len(numeric_features))
print('Categorical features:', categorical_features)
display(pd.DataFrame({'feature': feature_columns}))

## Attach SEC filing text

In [ ]:
import hashlib
import html as html_lib
import json
import re
import time
from collections.abc import Iterable

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


TEXT_COLUMN_CANDIDATES = [
    "sec_text",
    "filing_text",
    "document_text",
    "mda_text",
    "management_discussion_text",
    "risk_factors_text",
    "risk_text",
]


def normalize_accession(value: object) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def combine_available_text_columns(frame: pd.DataFrame) -> pd.Series:
    available = [
        column for column in TEXT_COLUMN_CANDIDATES
        if column in frame.columns
    ]
    if not available:
        return pd.Series("", index=frame.index, dtype="object")

    combined = (
        frame[available]
        .fillna("")
        .astype(str)
        .apply(
            lambda row: "\n\n".join(
                value.strip()
                for value in row
                if value and value.strip()
            ),
            axis=1,
        )
    )
    return combined


def discover_text_data_path() -> Path | None:
    candidates = [
        Path("/content/drive/MyDrive/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/drive/MyDrive/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("/content/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("sec_filing_text.parquet"),
        Path("sec_filing_text.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError("Text dataset must be a .parquet or .csv file.")


def merge_external_text(
    base: pd.DataFrame,
    text_frame: pd.DataFrame,
) -> pd.DataFrame:
    external = text_frame.copy()

    for frame in [base, external]:
        if "quarter_end" in frame.columns:
            frame["quarter_end"] = pd.to_datetime(
                frame["quarter_end"], errors="coerce"
            )
        if "filing_date" in frame.columns:
            frame["filing_date"] = pd.to_datetime(
                frame["filing_date"], errors="coerce"
            )
        if "accession_number" in frame.columns:
            frame["accession_number"] = frame[
                "accession_number"
            ].map(normalize_accession)
        if "cik" in frame.columns:
            frame["cik"] = frame["cik"].astype(str)
        if "ticker" in frame.columns:
            frame["ticker"] = frame["ticker"].astype(str)

    external["external_sec_text"] = combine_available_text_columns(
        external
    )

    if "accession_number" in base.columns and "accession_number" in external.columns:
        merge_keys = ["accession_number"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["cik", "quarter_end"]
    ):
        merge_keys = ["cik", "quarter_end"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["ticker", "quarter_end"]
    ):
        merge_keys = ["ticker", "quarter_end"]
    else:
        raise ValueError(
            "The text dataset must share accession_number, "
            "(cik, quarter_end), or (ticker, quarter_end) with the "
            "experiment dataset."
        )

    keep_columns = merge_keys + ["external_sec_text"]
    if "filing_date" in external.columns:
        keep_columns.append("filing_date")

    external = (
        external[keep_columns]
        .drop_duplicates(subset=merge_keys, keep="last")
    )

    merged = base.merge(
        external,
        on=merge_keys,
        how="left",
        suffixes=("", "_text_source"),
    )

    merged["sec_text"] = np.where(
        merged["sec_text"].fillna("").str.len()
        >= merged["external_sec_text"].fillna("").str.len(),
        merged["sec_text"].fillna(""),
        merged["external_sec_text"].fillna(""),
    )
    return merged


def clean_filing_html(raw_html: str) -> str:
    soup = BeautifulSoup(raw_html, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    # The project is using the narrative writing; large XBRL tables add
    # many repeated numbers and labels without much prose.
    for table in soup.find_all("table"):
        table.decompose()

    text = soup.get_text(" ")
    text = html_lib.unescape(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_longest_section(
    text: str,
    start_patterns: list[str],
    end_patterns: list[str],
    minimum_chars: int = 500,
    maximum_chars: int = 100_000,
) -> str:
    starts = []
    for pattern in start_patterns:
        starts.extend(re.finditer(pattern, text, flags=re.I))

    ends = []
    for pattern in end_patterns:
        ends.extend(re.finditer(pattern, text, flags=re.I))

    candidates: list[str] = []
    for start in starts:
        possible_ends = [
            end for end in ends
            if end.start() > start.end() + minimum_chars
        ]
        if not possible_ends:
            continue
        end = min(possible_ends, key=lambda match: match.start())
        candidate = text[start.start():end.start()].strip()
        if minimum_chars <= len(candidate) <= maximum_chars:
            candidates.append(candidate)

    return max(candidates, key=len) if candidates else ""


def extract_narrative_sections(clean_text: str) -> str:
    mda = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+7[\.\:\-\s]+management[’']?s?\s+discussion",
            r"\bitem\s+2[\.\:\-\s]+management[’']?s?\s+discussion",
        ],
        end_patterns=[
            r"\bitem\s+7a[\.\:\-\s]+",
            r"\bitem\s+8[\.\:\-\s]+financial",
            r"\bitem\s+3[\.\:\-\s]+quantitative",
            r"\bitem\s+4[\.\:\-\s]+controls",
        ],
    )

    risks = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+1a[\.\:\-\s]+risk\s+factors",
        ],
        end_patterns=[
            r"\bitem\s+1b[\.\:\-\s]+",
            r"\bitem\s+1c[\.\:\-\s]+",
            r"\bitem\s+2[\.\:\-\s]+",
        ],
    )

    sections = []
    if mda:
        sections.append("MANAGEMENT DISCUSSION AND ANALYSIS\n" + mda)
    if risks:
        sections.append("RISK FACTORS\n" + risks)

    if sections:
        return "\n\n".join(sections)[:MAX_DOCUMENT_CHARS]

    # Fallback when filing headings differ from the standard patterns.
    return clean_text[:MAX_DOCUMENT_CHARS]


def valid_sec_user_agent(user_agent: str) -> bool:
    lowered = user_agent.lower()
    return (
        "your name" not in lowered
        and "example.com" not in lowered
        and "@" in user_agent
        and len(user_agent.strip()) >= 8
    )


class EdgarTextDownloader:
    def __init__(self, user_agent: str, cache_dir: Path):
        if not valid_sec_user_agent(user_agent):
            raise ValueError(
                "Replace SEC_USER_AGENT with your real name and email "
                "before downloading from EDGAR."
            )

        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": user_agent,
                "Accept-Encoding": "gzip, deflate",
            }
        )
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.submission_cache: dict[str, dict[str, str]] = {}

    def get_json(self, url: str) -> dict:
        response = self.session.get(url, timeout=60)
        response.raise_for_status()
        time.sleep(0.20)
        return response.json()

    def get_text(self, url: str) -> str:
        response = self.session.get(url, timeout=90)
        response.raise_for_status()
        time.sleep(0.20)
        return response.text

    def _add_submission_rows(
        self,
        mapping: dict[str, str],
        payload: dict,
    ) -> None:
        accessions = payload.get("accessionNumber", [])
        primary_documents = payload.get("primaryDocument", [])
        for accession, document in zip(
            accessions, primary_documents, strict=False
        ):
            if accession and document:
                mapping[str(accession)] = str(document)

    def submission_map(self, cik: str) -> dict[str, str]:
        cik_key = str(int(float(cik))).zfill(10)
        if cik_key in self.submission_cache:
            return self.submission_cache[cik_key]

        payload = self.get_json(
            f"https://data.sec.gov/submissions/CIK{cik_key}.json"
        )
        mapping: dict[str, str] = {}
        self._add_submission_rows(
            mapping,
            payload.get("filings", {}).get("recent", {}),
        )

        # Older filings can be stored in additional submission JSON files.
        for file_info in payload.get("filings", {}).get("files", []):
            file_name = file_info.get("name")
            if not file_name:
                continue
            old_payload = self.get_json(
                f"https://data.sec.gov/submissions/{file_name}"
            )
            self._add_submission_rows(mapping, old_payload)

        self.submission_cache[cik_key] = mapping
        return mapping

    def primary_document(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        if supplied_document is not None and not pd.isna(supplied_document):
            supplied = str(supplied_document).strip()
            if supplied:
                return supplied

        mapping = self.submission_map(cik)
        document = mapping.get(accession)
        if document:
            return document

        # Final fallback: choose the largest non-index HTML document.
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        index_payload = self.get_json(
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/index.json"
        )
        items = (
            index_payload.get("directory", {}).get("item", [])
        )
        html_items = [
            item for item in items
            if str(item.get("name", "")).lower().endswith(
                (".htm", ".html")
            )
            and "-index." not in str(item.get("name", "")).lower()
            and "filingsummary" not in str(item.get("name", "")).lower()
        ]
        if not html_items:
            raise FileNotFoundError(
                f"No primary HTML document found for {accession}."
            )
        chosen = max(
            html_items,
            key=lambda item: int(item.get("size", 0) or 0),
        )
        return str(chosen["name"])

    def filing_text(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        accession = normalize_accession(accession)
        if accession is None:
            return ""

        cache_path = self.cache_dir / f"{accession}.txt"
        if cache_path.exists():
            return cache_path.read_text(
                encoding="utf-8", errors="ignore"
            )

        document = self.primary_document(
            cik, accession, supplied_document
        )
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        url = (
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/{document}"
        )
        raw_html = self.get_text(url)
        cleaned = clean_filing_html(raw_html)
        narrative = extract_narrative_sections(cleaned)
        cache_path.write_text(narrative, encoding="utf-8")
        return narrative


# Begin with any text already present in the experiment dataset.
model_data = model_data.copy()
model_data["sec_text"] = combine_available_text_columns(model_data)

# Merge a separate text table when configured or automatically found.
resolved_text_path = (
    TEXT_DATA_PATH
    if TEXT_DATA_PATH is not None
    else discover_text_data_path()
)
if resolved_text_path is not None:
    if not resolved_text_path.exists():
        raise FileNotFoundError(
            f"TEXT_DATA_PATH does not exist: {resolved_text_path}"
        )
    external_text = read_table(resolved_text_path)
    model_data = merge_external_text(model_data, external_text)
    print("Merged SEC text dataset:", resolved_text_path)

# Download only missing filing text, using one request per unique filing.
missing_text = model_data["sec_text"].fillna("").str.len() < MIN_TEXT_CHARS
can_download = (
    DOWNLOAD_SEC_TEXT_FROM_EDGAR
    and missing_text.any()
    and {"cik", "accession_number"}.issubset(model_data.columns)
    and valid_sec_user_agent(SEC_USER_AGENT)
)

if can_download:
    downloader = EdgarTextDownloader(
        SEC_USER_AGENT,
        TEXT_CACHE_DIR / "filings",
    )

    unique_filings = (
        model_data.loc[
            missing_text,
            [
                column
                for column in [
                    "cik",
                    "accession_number",
                    "primary_document",
                ]
                if column in model_data.columns
            ],
        ]
        .dropna(subset=["cik", "accession_number"])
        .drop_duplicates(subset=["cik", "accession_number"])
    )

    downloaded_text: dict[tuple[str, str], str] = {}
    failures = []

    for _, filing in tqdm(
        unique_filings.iterrows(),
        total=len(unique_filings),
        desc="Downloading SEC filings",
    ):
        cik = str(filing["cik"])
        accession = normalize_accession(filing["accession_number"])
        if accession is None:
            continue
        supplied_document = (
            filing.get("primary_document")
            if "primary_document" in filing.index
            else None
        )
        try:
            downloaded_text[(cik, accession)] = (
                downloader.filing_text(
                    cik,
                    accession,
                    supplied_document,
                )
            )
        except Exception as exc:
            failures.append(
                {
                    "cik": cik,
                    "accession_number": accession,
                    "error": str(exc),
                }
            )

    row_keys = list(
        zip(
            model_data["cik"].astype(str),
            model_data["accession_number"].map(normalize_accession),
        )
    )
    downloaded_series = pd.Series(
        [
            downloaded_text.get(key, "")
            for key in row_keys
        ],
        index=model_data.index,
    )
    replace_mask = (
        model_data["sec_text"].fillna("").str.len()
        < downloaded_series.str.len()
    )
    model_data.loc[replace_mask, "sec_text"] = downloaded_series[
        replace_mask
    ]

    if failures:
        failure_frame = pd.DataFrame(failures)
        failure_frame.to_csv(
            TEXT_CACHE_DIR / "edgar_download_failures.csv",
            index=False,
        )
        print(
            f"{len(failures)} filing downloads failed. Details were saved "
            "to edgar_download_failures.csv."
        )

elif missing_text.any():
    print(
        "SEC text is not yet available for all rows.\n"
        "Use one of these options:\n"
        "  1. Put sec_text, filing_text, mda_text, or risk_factors_text "
        "in the experiment dataset.\n"
        "  2. Set TEXT_DATA_PATH to a matching text parquet/CSV.\n"
        "  3. Replace SEC_USER_AGENT with your real name and email so "
        "the notebook can download filings from EDGAR."
    )

# Prevent obvious timing leakage when both timestamps are available.
if (
    "filing_date" in model_data.columns
    and "feature_cutoff_date" in model_data.columns
):
    filing_dates = pd.to_datetime(
        model_data["filing_date"], errors="coerce"
    )
    cutoff_dates = pd.to_datetime(
        model_data["feature_cutoff_date"], errors="coerce"
    )
    late_text = (
        filing_dates.notna()
        & cutoff_dates.notna()
        & (filing_dates > cutoff_dates)
    )
    if late_text.any():
        print(
            f"Removed text from {late_text.sum()} rows because the filing "
            "date was after the feature cutoff date."
        )
        model_data.loc[late_text, "sec_text"] = ""

model_data["text_characters"] = (
    model_data["sec_text"].fillna("").str.len()
)
model_data["has_sec_text"] = (
    model_data["text_characters"] >= MIN_TEXT_CHARS
)

text_coverage = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        rows_with_text=("has_sec_text", "sum"),
        median_text_characters=("text_characters", "median"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
text_coverage["coverage"] = (
    text_coverage["rows_with_text"] / text_coverage["rows"]
)
display(text_coverage)

## Keep identical text-available rows

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

text_model_data = model_data[
    model_data['has_sec_text'] & model_data['split'].isin(['train','validation','test'])
].copy()
for s in ['train','validation','test']:
    sub = text_model_data[text_model_data.split == s]
    if sub.empty or sub.target_clean.nunique() < 2:
        raise ValueError(f'{s} needs text rows from both classes. Add more text/data or reduce NEUTRAL_BAND.')
print(text_model_data.groupby('split').agg(rows=('target_clean','size'), companies=('cik','nunique'), classes=('target_clean','nunique')))
TEXT_MODELS_READY = True

## Sentence Transformer embeddings

In [ ]:
def chunk_document(
    text: str,
    chunk_words: int = CHUNK_WORDS,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
    maximum_chunks: int = MAX_CHUNKS_PER_FILING,
) -> list[str]:
    words = str(text).split()
    if not words:
        return []

    step = max(1, chunk_words - overlap_words)
    chunks = [
        " ".join(words[start:start + chunk_words])
        for start in range(0, len(words), step)
        if len(words[start:start + chunk_words]) >= 30
    ]

    if not chunks:
        return [" ".join(words)]

    if len(chunks) > maximum_chunks:
        selected_indices = np.linspace(
            0,
            len(chunks) - 1,
            maximum_chunks,
            dtype=int,
        )
        chunks = [chunks[index] for index in selected_indices]

    return chunks


def text_hash(text: str) -> str:
    payload = (
        TEXT_EMBEDDING_MODEL
        + "\n"
        + str(MAX_CHUNKS_PER_FILING)
        + "\n"
        + str(text)
    )
    return hashlib.sha1(
        payload.encode("utf-8", errors="ignore")
    ).hexdigest()


def build_sentence_embeddings(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    from sentence_transformers import SentenceTransformer

    working = frame.copy()
    working["text_hash"] = working["sec_text"].map(text_hash)

    safe_model_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        TEXT_EMBEDDING_MODEL,
    )
    cache_path = (
        TEXT_CACHE_DIR
        / f"document_embeddings_{safe_model_name}.parquet"
    )

    cached = pd.DataFrame()
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)

    cached_hashes = (
        set(cached["text_hash"])
        if not cached.empty and "text_hash" in cached.columns
        else set()
    )

    unique_documents = (
        working[["text_hash", "sec_text"]]
        .drop_duplicates("text_hash")
    )
    missing_documents = unique_documents[
        ~unique_documents["text_hash"].isin(cached_hashes)
    ]

    if not missing_documents.empty:
        encoder = SentenceTransformer(TEXT_EMBEDDING_MODEL)

        flat_chunks: list[str] = []
        owners: list[str] = []
        for row in missing_documents.itertuples(index=False):
            chunks = chunk_document(row.sec_text)
            flat_chunks.extend(chunks)
            owners.extend([row.text_hash] * len(chunks))

        if not flat_chunks:
            raise ValueError("No valid text chunks were produced.")

        chunk_embeddings = encoder.encode(
            flat_chunks,
            batch_size=TEXT_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        chunk_frame = pd.DataFrame(chunk_embeddings)
        chunk_frame.insert(0, "text_hash", owners)

        document_embeddings = (
            chunk_frame.groupby("text_hash", sort=False)
            .mean()
            .reset_index()
        )

        embedding_columns = [
            column
            for column in document_embeddings.columns
            if column != "text_hash"
        ]
        matrix = document_embeddings[embedding_columns].to_numpy()
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        matrix = matrix / np.maximum(norms, 1e-12)
        document_embeddings[embedding_columns] = matrix

        if cached.empty:
            cached = document_embeddings
        else:
            cached = pd.concat(
                [cached, document_embeddings],
                ignore_index=True,
            ).drop_duplicates("text_hash", keep="last")

        cached.to_parquet(cache_path, index=False)

    embedding_columns_in_cache = [
        column for column in cached.columns
        if column != "text_hash"
    ]
    renamed = {
        column: f"text_embedding_{int(column):03d}"
        for column in embedding_columns_in_cache
    }
    cached = cached.rename(columns=renamed)
    embedding_columns = list(renamed.values())

    working = working.merge(
        cached,
        on="text_hash",
        how="left",
    )
    return working, embedding_columns


if TEXT_MODELS_READY:
    text_model_data, text_embedding_columns = (
        build_sentence_embeddings(text_model_data)
    )
    print(
        "Sentence embedding dimensions:",
        len(text_embedding_columns),
    )
else:
    text_embedding_columns = []

## FinBERT features

In [ ]:
def build_finbert_sentiment_features(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    import torch
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
    )

    output = frame.copy()
    feature_names = [
        "finbert_positive_mean",
        "finbert_neutral_mean",
        "finbert_negative_mean",
        "finbert_negative_max",
        "finbert_negative_std",
        "finbert_positive_minus_negative",
    ]

    if not RUN_FINBERT_SENTIMENT:
        for feature in feature_names:
            output[feature] = 0.0
        return output, []

    tokenizer = AutoTokenizer.from_pretrained(
        FINBERT_MODEL_NAME
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL_NAME
    )
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model.to(device)
    model.eval()

    id_to_label = {
        int(index): str(label).lower()
        for index, label in model.config.id2label.items()
    }

    rows = []
    for text in tqdm(
        output["sec_text"],
        desc="FinBERT sentiment",
    ):
        chunks = chunk_document(
            text,
            chunk_words=180,
            overlap_words=30,
            maximum_chunks=12,
        )
        probabilities = []

        for start in range(0, len(chunks), TEXT_BATCH_SIZE):
            batch = chunks[start:start + TEXT_BATCH_SIZE]
            tokens = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)

            with torch.no_grad():
                logits = model(**tokens).logits
                batch_probabilities = torch.softmax(
                    logits, dim=-1
                ).cpu().numpy()
            probabilities.append(batch_probabilities)

        matrix = np.vstack(probabilities)
        label_columns = {
            label: matrix[:, index]
            for index, label in id_to_label.items()
        }
        positive = label_columns.get(
            "positive", np.zeros(len(matrix))
        )
        neutral = label_columns.get(
            "neutral", np.zeros(len(matrix))
        )
        negative = label_columns.get(
            "negative", np.zeros(len(matrix))
        )

        rows.append(
            {
                "finbert_positive_mean": positive.mean(),
                "finbert_neutral_mean": neutral.mean(),
                "finbert_negative_mean": negative.mean(),
                "finbert_negative_max": negative.max(),
                "finbert_negative_std": negative.std(),
                "finbert_positive_minus_negative": (
                    positive.mean() - negative.mean()
                ),
            }
        )

    sentiment_frame = pd.DataFrame(
        rows,
        index=output.index,
    )
    for feature in feature_names:
        output[feature] = sentiment_frame[feature]

    return output, feature_names


if TEXT_MODELS_READY:
    text_model_data, finbert_feature_columns = (
        build_finbert_sentiment_features(text_model_data)
    )
else:
    finbert_feature_columns = []

print("FinBERT features included:", finbert_feature_columns)

## Train combined numerical + textual models

In [ ]:
def choose_threshold(y_true, probabilities):
    scores=[balanced_accuracy_score(y_true,(probabilities>=t).astype(int)) for t in THRESHOLD_GRID]
    i=int(np.argmax(scores)); return float(THRESHOLD_GRID[i]), float(scores[i])


def evaluation_row(name,y,p,pred,base_p):
    base=np.full(len(y),base_p); bb=brier_score_loss(y,base); b=brier_score_loss(y,p)
    return {'model':name,'rows':len(y),'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),'macro_f1':f1_score(y,pred,average='macro',zero_division=0),'acceleration_recall':recall_score(y,pred,pos_label=1,zero_division=0),'deceleration_recall':recall_score(y,pred,pos_label=0,zero_division=0),'brier_score':b,'brier_skill_score':1-b/bb if bb>0 else np.nan,'roc_auc':roc_auc_score(y,p) if pd.Series(y).nunique()==2 else np.nan}


def make_combined_pipeline(mode,C,class_weight,train_rows,min_df):
    branches=[]
    if numeric_features:
        branches.append(('financial_numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),numeric_features))
    if categorical_features:
        branches.append(('financial_categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical_features))
    if mode=='tfidf':
        branches.append(('tfidf',TfidfVectorizer(lowercase=True,strip_accents='unicode',stop_words='english',ngram_range=(1,2),min_df=min_df,max_df=0.98,max_features=20000,sublinear_tf=True),'sec_text'))
    if mode in {'sentence','sentence_finbert'}:
        ncomp=max(1,min(TEXT_PCA_COMPONENTS,len(text_embedding_columns),train_rows-1))
        branches.append(('sentence_embeddings',Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value=0.0)),('scaler',StandardScaler()),('pca',PCA(n_components=ncomp,random_state=RANDOM_STATE))]),text_embedding_columns))
    if mode in {'finbert','sentence_finbert'}:
        branches.append(('finbert',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),finbert_feature_columns))
    prep=ColumnTransformer(branches,remainder='drop',verbose_feature_names_out=False)
    solver='liblinear' if mode=='tfidf' else 'lbfgs'
    return Pipeline([('preprocessor',prep),('classifier',LogisticRegression(C=C,penalty='l2',class_weight=class_weight,solver=solver,max_iter=5000,random_state=RANDOM_STATE))])

train=text_model_data[text_model_data.split=='train'].copy(); val=text_model_data[text_model_data.split=='validation'].copy(); test=text_model_data[text_model_data.split=='test'].copy(); train_val=pd.concat([train,val],ignore_index=True)
y_train,y_val,y_test=train.target_clean,val.target_clean,test.target_clean;base_p=float(train_val.target_clean.mean());min_df=1 if len(train)<100 else 2

specs={
 'Numerical + TF-IDF + Logistic Regression':'tfidf',
 'Numerical + Sentence Transformer + Logistic Regression':'sentence',
 'Numerical + FinBERT + Logistic Regression':'finbert',
 'Numerical + Sentence Transformer + FinBERT + Logistic Regression':'sentence_finbert',
}
rows=[];selection=[];models={};preds=test[[c for c in ['cik','ticker','company_name','quarter_end','future_growth_change','target_clean'] if c in test.columns]].copy()
prior=np.full(len(test),base_p);rows.append(evaluation_row('Prior-probability baseline',y_test,prior,(prior>=0.5).astype(int),base_p))
for name,mode in specs.items():
    best=None
    for w in [None,'balanced']:
        for C in C_GRID:
            pipe=make_combined_pipeline(mode,C,w,len(train),min_df);pipe.fit(train,y_train);p=pipe.predict_proba(val)[:,1];b=brier_score_loss(y_val,p);t,bal=choose_threshold(y_val,p);cand={'C':C,'class_weight':w,'brier':b,'threshold':t,'val_bal':bal}
            if best is None or b<best['brier']:best=cand
    final=make_combined_pipeline(mode,best['C'],best['class_weight'],len(train_val),min_df);final.fit(train_val,train_val.target_clean);p=final.predict_proba(test)[:,1];pred=(p>=best['threshold']).astype(int);row=evaluation_row(name,y_test,p,pred,base_p);row.update({'selected_C':best['C'],'class_weight':str(best['class_weight']),'threshold':best['threshold'],'validation_brier':best['brier']});rows.append(row);selection.append({'model':name,**best});models[name]=final;preds[f'{name}_probability']=p;preds[f'{name}_prediction']=pred
results=pd.DataFrame(rows).sort_values(['brier_score','balanced_accuracy'],ascending=[True,False]);selection=pd.DataFrame(selection).sort_values('brier');display(results)
results.to_csv(OUTPUT_DIR/'combined_test_results.csv',index=False);selection.to_csv(OUTPUT_DIR/'combined_validation_selection.csv',index=False);preds.to_csv(OUTPUT_DIR/'combined_test_predictions.csv',index=False)
with pd.ExcelWriter(OUTPUT_DIR/'combined_branch_results.xlsx',engine='openpyxl') as writer:
    results.to_excel(writer,sheet_name='Model_Results',index=False);selection.to_excel(writer,sheet_name='Validation_Selection',index=False);preds.to_excel(writer,sheet_name='Test_Predictions',index=False)
joblib.dump(models,OUTPUT_DIR/'combined_final_models.joblib');print('Saved to:',OUTPUT_DIR)